# Titin Insilico variants

In [1]:
from cellpose import models, core, io, plot
import matplotlib.pyplot as plt
import pandas as pd
import cellpose.utils as utils_cell
from parsho.img_utils import *
from parsho.segmentation import *
from parsho.maskfilters import *
from parsho.plotting import *
import glob

In [2]:
# cellpose SAM parameters
flow_threshold = 0.0 
cellprob_threshold = -2.0
tile_norm_blocksize = 0

# Load example image and extract channels
f = "/mnt/light-roast/ograciac/MorroCells/data3/C3-E3_117plate4-1.tif"
pos_img_cell = io.imread(f)

f = "/mnt/light-roast/ograciac/MorroCells/data3/C2-E3_117plate4-1.tif"
pos_img_agregates = io.imread(f)

f = "/mnt/light-roast/ograciac/MorroCells/data3/C1-E3_117plate4-1.tif"
pos_img_nuclei = io.imread(f)

f = "/mnt/light-roast/ograciac/MorroCells/data3/C3-G8_228plate5-1.tif"
neg_img_cell = io.imread(f)

f = "/mnt/light-roast/ograciac/MorroCells/data3/C2-G8_228plate5-1.tif"
neg_img_agregates = io.imread(f)

f = "/mnt/light-roast/ograciac/MorroCells/data3/C1-G8_228plate5-1.tif"
neg_img_nuclei = io.imread(f)

In [3]:
# Run cellpose
model = models.CellposeModel(gpu=True)

pos_masks, pos_flows, pos_styles = model.eval(pos_img_cell, batch_size=32, flow_threshold=flow_threshold, cellprob_threshold=cellprob_threshold,
                                  normalize={"tile_norm_blocksize": tile_norm_blocksize})

neg_masks, neg_flows, neg_styles = model.eval(neg_img_cell, batch_size=32, flow_threshold=flow_threshold, cellprob_threshold=cellprob_threshold,
                                  normalize={"tile_norm_blocksize": tile_norm_blocksize})

In [4]:
# Extract masks for example channels
pos_agg_binary, pos_agg_labels, p_agg_t = extract_aggregate_masks(
    aggregate_channel=pos_img_agregates,  
    cell_masks=pos_masks,
    method="otsu",
    min_size_px=2,
    scale=1,
)

neg_agg_binary, neg_agg_labels, n_agg_t = extract_aggregate_masks(
    aggregate_channel=neg_img_agregates,   
    cell_masks=neg_masks,
    method="otsu",
    min_size_px=2,
    scale=1,
)

pos_nuc_binary, pos_nuc_labels, p_t = extract_aggregate_masks(
    aggregate_channel=pos_img_nuclei,   
    cell_masks=pos_masks,
    method="otsu",
    min_size_px=5,
    scale=1,
)

neg_nuc_binary, neg_nuc_labels, n_t = extract_aggregate_masks(
    aggregate_channel=neg_img_nuclei,    
    cell_masks=neg_masks,
    method="otsu",
    min_size_px=5,
    scale=1,
)


In [5]:
# Find optimal threshold
optimal_thresh, scores = find_optimal_threshold(
    pos_img_agregates,
    pos_masks,
    neg_img_agregates,
    neg_masks,
    pos_nuc_binary,
    neg_nuc_binary,
    t_min=n_agg_t,
    t_max=p_agg_t,
)

print(f"Optimal threshold: {optimal_thresh}")
print(f"Score at optimal threshold: {scores}")

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
Optimal threshold: 1539.5338134765625
Score at optimal threshold: 442


In [6]:
# Variant name mapping to titin variants
variant_names = {
    "E9": "Fn3-119_D31735R",
    "D4": "Fn3-119_E31759F",
    "D5": "Fn3-119_E31777V",
    "D3": "Fn3-119_Frameshift",
    "F6": "Fn3-119_G31710H",
    "E10": "Fn3-119_G31737C",
    "E11": "Fn3-119_I31740C",
    "G1": "Fn3-119_I31757T",
    "D6": "Fn3-119_MAYBE_E31795D",
    "G9": "Fn3-119_maybe_InDel",
    "G3": "Fn3-119_P31793R",
    "D2": "Fn3-119_S31708Y",
    "G8": "Fn3-119_T31750K",
    "H7": "Fn3-119_V31745L",
    "G10": "Fn3-119_V31800T",
    "E8": "Fn3-119_WT",
    "D11": "Fn3-27_A19253M",
    "B8": "Fn3-27_C19201D",
    "B9": "Fn3-27_E19209S",
    "A3": "Fn3-27_E19261N",
    "B5": "Fn3-27_E19271Y",
    "A2": "Fn3-27_F19255N",
    "A5": "Fn3-27_I19278W",
    "B11": "Fn3-27_K19223N",
    "B12": "Fn3-27_R19224P",
    "B4": "Fn3-27_R19257E",
    "B10": "Fn3-27_S19214W",
    "A4": "Fn3-27_S19273P",
    "B6": "Fn3-27_S19273W",
    "A1": "Fn3-27_T19189M",
    "D7": "Fn3-27_WT",
    "D10": "Fn3-27_WT",
    "B7": "Fn3-27_WT_T19197W",
    "D9": "Fn3-27_Y19236T",
    "H8": "Fn3-45_A21736M",
    "C3": "Fn3-45_A21740K",
    "B1": "Fn3-45_D21686K",
    "G7": "Fn3-45_D21711C",
    "H11": "Fn3-45_Del21743-45",
    "A6": "Fn3-45_F21677G",
    "F11": "Fn3-45_I21692S",
    "H9": "Fn3-45_K21739V",
    "F8": "Fn3-45_P21683E",
    "D12": "Fn3-45_R21667P",
    "C2": "Fn3-45_R21701F",
    "B2": "Fn3-45_S21731D",
    "H10": "Fn3-45_S21742G",
    "C1": "Fn3-45_WT",
    "F7": "Fn3-45_WT",
    "A12": "K136H",
    "G11": "M10_D35909M",
    "H3": "M10_D35951M",
    "E5": "M10_D35952F",
    "E2": "M10_E35927A",
    "H2": "M10_E35948I",
    "E3": "M10_F35945E",
    "G4": "M10_I35908W",
    "G5": "M10_K35912",
    "H1": "M10_K35936V",
    "E6": "M10_K35963W",
    "C9": "M10_N35949W",
    "E4": "M10_N35949W",
    "F1": "M10_S35979Y",
    "E1": "M10_T35914P",
    "G6": "M10_T35921I",
    "H6": "M10_T35983V_(MAYBE_ALSO_T35915P)",
    "H4": "M10_V35961H_(MAYBE_ALSO_T35915V)",
    "H5": "M10_WT",
    "A11": "No_DNA",
    "G12": "No_DNA",
    "F10": "No_sequence",
    "F12": "No_sequence",
    "G2": "No_sequence",
    "C4": "Z2_A142D",
    "D1": "Z2_A189E",
    "C6": "Z2_D149I",
    "F2": "Z2_D149I",
    "C5": "Z2_D149Y",
    "C7": "Z2_Duplication",
    "F3": "Z2_E165W",
    "F4": "Z2_G172Q",
    "F5": "Z2_L192E_(NOT_L192Q)",
    "A8": "Z2_M113W",
    "H12": "Z2_P104C",
    "A9": "Z2_Q117W",
    "A10": "Z2_Q124C",
    "A7": "Z2_R109T",
    "C10": "Z2_S187Q",
    "C11": "Z2_WT",
    "C8": "Z2_Y138P",
    "B3": "NA_B3",
    "C12": "NA_C12",
    "D8": "NA_D8",
    "E7": "NA_E7",
    "E12": "NA_E12",
    "F9": "NA_F9",
    "starve":"starve",
    "starve-rapa":"starve-rapa",
    "ebss-rapa":"ebss-rapa",
    "rapa":"rapa",
    "ebss":"ebss",
    "untrated":"untrated"
}


In [ ]:
# ------------------------------------------------------------------ #
# Experiment configuration  (edit these for each run)
# ------------------------------------------------------------------ #
 
DATA_DIR    = Path("/mnt/light-roast/ograciac/MorroCells/data2")
OUT_DIR     = Path("/mnt/light-roast/ograciac/MorroCells/results_morro/channels")
CSV_OUT     = Path("/mnt/light-roast/ograciac/MorroCells/results_morro/final_data.csv")

# ------------------------------------------------------------------ #
# Experiment-specific helpers
# ------------------------------------------------------------------ #
 
def parse_name_and_well(filepath: str) -> tuple[str, str]:
    """Extract the shared image name and well ID from a file path."""
    fname = Path(filepath).name          # e.g. 'C1_A1_001.tif'
    name  = fname[3:]                    # drop leading channel prefix
    well  = fname.split("_")[0][3:]      # e.g. 'A1'
    return name, well
 
 
def load_image_set(data_dir: Path, name: str) -> tuple:
    """Load the three channel images for a given image name."""
    img_aggregates = io.imread(data_dir / f"C2-{name}")
    img_cell       = io.imread(data_dir / f"C3-{name}")
    img_nuclei     = io.imread(data_dir / f"C1-{name}")
    return img_cell, img_aggregates, img_nuclei
 
 

OUT_DIR.mkdir(parents=True, exist_ok=True)

files_done: list[str] = []
records: list[dict]   = []
file_num = 0

all_files = glob.glob(str(DATA_DIR / "*.tif"))

for ii, f in enumerate(all_files):
    print(f"{ii} of {len(all_files)}: {f}")

    name, well = parse_name_and_well(f)
    if name in files_done:
        continue

    files_done.append(name)
    file_num += 1

    # ---- Load -------------------------------------------------------- #
    img_cell, img_aggregates, img_nuclei = load_image_set(DATA_DIR, name)

    # ---- Segment cells ----------------------------------------------- #
    masks, flows, _ = model.eval(
        img_cell,
        batch_size=32,
        flow_threshold=flow_threshold,
        cellprob_threshold=cellprob_threshold,
        normalize={"tile_norm_blocksize": tile_norm_blocksize},
    )
    plot_segmentation_result(
        img_cell, masks, flows[0],
        save_path=OUT_DIR / f"cell_masks_{name}",
    )

    # ---- Extract channel masks --------------------------------------- #
    agg_binary, agg_labels = re_threshold_masks(
        aggregate_channel=img_aggregates,
        cell_masks=masks,
        min_size_px=2,
        thresh=optimal_thresh,
    )
    nuc_binary, nuc_labels, _ = extract_aggregate_masks(
        aggregate_channel=img_nuclei,
        cell_masks=masks,
        method="otsu",
        min_size_px=5,
        scale=1,
    )
    trans_binary, trans_labels, _ = extract_aggregate_masks(
        aggregate_channel=img_aggregates,
        cell_masks=masks,
        method="otsu",
        min_size_px=2,
        scale=1,
    )

    # ---- Post-process masks ----------------------------------------- #
    agg_binary, agg_labels = subtract_nuclear_from_aggregate(
        agg_binary, agg_labels, nuc_binary
    )

    outlines = utils_cell.masks_to_outlines(masks)
    overlay  = build_overlay(masks, nuc_labels, agg_labels, outlines)

    plot_aggregate_channel(agg_binary, save_path=f"/mnt/light-roast/ograciac/MorroCells/results_morro/channels/agg_masks_{name}")
    plot_aggregate_channel(nuc_binary, save_path=f"/mnt/light-roast/ograciac/MorroCells/results_morro/channels/nuc_masks_{name}")
    plot_aggregate_channel(trans_binary, save_path=f"/mnt/light-roast/ograciac/MorroCells/results_morro/channels/trans_masks_{name}")
    plot_aggregate_channel_color(overlay, save_path=f"/mnt/light-roast/ograciac/MorroCells/results_morro/channels/overlayed_{name}")

    # ---- Filter transfected cells ------------------------------------ #
    transfected_labels = filter_transfected_cells(masks, trans_labels, nuc_labels)
    filtered_overlay   = mask_overlay_to_transfected(overlay, masks, transfected_labels)

    # ---- Compute metrics --------------------------------------------- #
    text_mapping = {}
    for jj, metrics in enumerate(compute_cell_metrics(masks, nuc_labels, agg_labels, transfected_labels)):
        records.append({
            "variant":   variant_names.get(well, well),
            "well":      well,
            "File Name": name,
            "Img number": file_num,
            "cell num":  jj,
            "cell area": metrics.cell_area,
            "agg area":  metrics.agg_area,
            "nuc area":  metrics.nuc_area,
            "jaccard":   metrics.jaccard,
            "cell aspect ratio": metrics.cell_aspect_ratio,
            "cell circularity": metrics.cell_circularity,
            "agg aspect ratio": metrics.agg_aspect_ratio,
            "agg circularity": metrics.agg_circularity
        })
        text_mapping[metrics.label] = jj

    # ---- Save filtered and labelled figure --- #
    plot_aggregate_channel_color_labelled(filtered_overlay, masks, text_mapping, save_path=f"/mnt/light-roast/ograciac/MorroCells/results_morro/channels/overlayed_trans_nuc_filter_{name}")

df = pd.DataFrame(records)
df.to_csv(CSV_OUT, index=False)
print(f"\nSaved {len(df)} rows → {CSV_OUT}")


0 of 2574: /mnt/light-roast/ograciac/MorroCells/data2/C3-B8_031plate2.tif
1 of 2574: /mnt/light-roast/ograciac/MorroCells/data2/C2-D4_079plate1.tif
2 of 2574: /mnt/light-roast/ograciac/MorroCells/data2/C1-B9_047plate4.tif
3 of 2574: /mnt/light-roast/ograciac/MorroCells/data2/C2-A7_017plate4.tif
4 of 2574: /mnt/light-roast/ograciac/MorroCells/data2/C3-F5_183plate5.tif
5 of 2574: /mnt/light-roast/ograciac/MorroCells/data2/C2-C1_058plate4.tif
6 of 2574: /mnt/light-roast/ograciac/MorroCells/data2/C2-G8_228plate5.tif
7 of 2574: /mnt/light-roast/ograciac/MorroCells/data2/C2-D12_134plate5.tif
8 of 2574: /mnt/light-roast/ograciac/MorroCells/data2/C2-B9_054plate5.tif
9 of 2574: /mnt/light-roast/ograciac/MorroCells/data2/C1-D7_119plate5.tif
10 of 2574: /mnt/light-roast/ograciac/MorroCells/data2/C2-C9_078plate4.tif
11 of 2574: /mnt/light-roast/ograciac/MorroCells/data2/C3-F1_058plate3.tif
12 of 2574: /mnt/light-roast/ograciac/MorroCells/data2/C1-A10_019plate2.tif
13 of 2574: /mnt/light-roast/ogra